In [49]:
# depenencies
from astropy.io import fits
import glob
import os
import pandas as pd
import numpy as np
#import 
from astropy.stats import sigma_clipped_stats
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.wcs import WCS
from astropy.io import fits
import glob
import os
from scipy.stats import scoreatpercentile
import astropy.units as u
from scipy import stats

from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib import pyplot as plt
import numpy as np
from matplotlib import colors


from astropy.nddata import CCDData
#from ccdproc import ImageFileCollection, Combiner, combine
from ccdproc import wcs_project


#### Making catalog from files to get galaxy name and file for work. Don't run and overwrite custom galaxy catalog stuff if don't need to. Custom galaxy parameters such as x/ycent go directly in .csv.

In [73]:
file_list = glob.glob("../../Data/Capstone/*.fits.fz", recursive=True) # grab file list

modified_names = [filename.replace('../../Data/Capstone/', '').replace('-image-g.fits.fz', '') for filename in file_list] #strip badness so its just name of galaxy

# throw in dataframe for reading into code
catalog = pd.DataFrame({
    'file': file_list,
    'name': modified_names
})
# Display the DataFrame
print(catalog)

# save dataframe
catalog.to_csv("capstone_catalog.csv", index=False)

                                            file      name
0   ../../Data/Capstone/Z 190-31-image-g.fits.fz  Z 190-31
1   ../../Data/Capstone/NGC 5169-image-g.fits.fz  NGC 5169
2   ../../Data/Capstone/NGC 6643-image-g.fits.fz  NGC 6643
3     ../../Data/Capstone/IC 749-image-g.fits.fz    IC 749
4        ../../Data/Capstone/M95-image-g.fits.fz       M95
5      ../../Data/Capstone/NGC 1-image-g.fits.fz     NGC 1
6   ../../Data/Capstone/NGC 1094-image-g.fits.fz  NGC 1094
7   ../../Data/Capstone/NGC 5440-image-g.fits.fz  NGC 5440
8   ../../Data/Capstone/NGC 1087-image-g.fits.fz  NGC 1087
9    ../../Data/Capstone/NGC 253-image-g.fits.fz   NGC 253
10       ../../Data/Capstone/M99-image-g.fits.fz       M99
11  ../../Data/Capstone/NGC 5633-image-g.fits.fz  NGC 5633
12    ../../Data/Capstone/NGC 47-image-g.fits.fz    NGC 47
13  ../../Data/Capstone/NGC 4711-image-g.fits.fz  NGC 4711


#### Reading in catalog

In [100]:
catalog = pd.read_csv("capstone_catalog.csv")

In [101]:
display(catalog)

,file,name,xcen,ycen,sma,smina,ellip,PA
0,../../Data/Capstone/Z 190-31-image-g.fits.fz,Z 190-31,NaN,NaN,NaN,NaN,NaN,NaN
1,../../Data/Capstone/NGC 5169-image-g.fits.fz,NGC 5169,NaN,NaN,NaN,NaN,NaN,NaN
2,../../Data/Capstone/NGC 6643-image-g.fits.fz,NGC 6643,NaN,NaN,NaN,NaN,NaN,NaN
3,../../Data/Capstone/IC 749-image-g.fits.fz,IC 749,NaN,NaN,NaN,NaN,NaN,NaN
4,../../Data/Capstone/M95-image-g.fits.fz,M95,NaN,NaN,NaN,NaN,NaN,NaN
5,../../Data/Capstone/NGC 1-image-g.fits.fz,NGC 1,NaN,NaN,NaN,NaN,NaN,NaN
6,../../Data/Capstone/NGC 1094-image-g.fits.fz,NGC 1094,NaN,NaN,NaN,NaN,NaN,NaN
7,../../Data/Capstone/NGC 5440-image-g.fits.fz,NGC 5440,NaN,NaN,NaN,NaN,NaN,NaN
8,../../Data/Capstone/NGC 1087-image-g.fits.fz,NGC 1087,NaN,NaN,NaN,NaN,NaN,NaN
9,../../Data/Capstone/NGC 253-image-g.fits.fz,NGC 253,NaN,NaN,NaN,NaN,NaN,NaN


Custom guesses for each galaxy in the catalog are xcen/ycen (center of centroid in pix), sma and smina (semi major and minor axes for ellipse in pix), ellip (the ellipticity), and PA (the physical angle in deg from x direction).

In [117]:
# Test only the first object
files_to_run = catalog.iloc[[11]]
# Or process everything
# files_to_run = catalog
[ellipsefitting(row, plot_opt = True) for _, row in files_to_run.iterrows()]

Galaxy name: NGC 5633	Galaxy filepath:../../Data/Capstone/NGC 5633-image-g.fits.fz
152.0 0.296052631578947


[None]

#### Helper Functions

In [112]:
def imdisplay(image, v1perc=50, v2perc=99.99, logscale=True):
    '''
    display an image 
    OPTIONAL KEYWORD PARAMETERS
    v1perc: one end of the colormap assigned to the v1perc percent lowest flux 
    v2perc: the other end of the colormap assigned to the v2perc percent highest flux    
    '''
    fig,ax = plt.subplots(figsize=(6,6))
    # make sure image is an np array
    nimage = np.array(image)
    # determine the pixel values at the 10th and 95th percentile

    # display using imshow
    #
    # you can play with alternate cmaps in the function below, such as "viridis" or "gray"
    # The 'gray_r' color map reverses the color-scale so that dark display pixels are the brightest in the image
    #
    # vmin and vmax set the min and max pixel values that
    # will be mapped to the extremes of the colormap
    if (logscale):
        #v1 = scoreatpercentile(nimage,v1perc)
        #v2 = scoreatpercentile(nimage,v2perc)
        #print(v1,v2)
        v1 = 0.0001 # debug custom values
        v2 = 0.3 # debug custom values
        im = ax.imshow(nimage,cmap='gray_r',aspect='equal',norm=colors.LogNorm(vmin=v1,vmax=v2),origin='lower')
    else:
        v1 = v1perc
        v2 = v2perc
        print(v1,v2)
        im = ax.imshow(nimage,cmap='gray_r',aspect='equal',vmin=v1,vmax=v2,origin='lower')
        
    cbar = fig.colorbar(im, ax=ax, fraction=0.04)
    cbar.ax.set_ylabel('counts')
    return fig,ax


In [113]:
def imdisplay2(image, v1perc=50, v2perc=95, logscale=True):
    '''
    display two copies of an image side-by-side
    OPTIONAL KEYWORD PARAMETERS
    v1perc: one end of the colormap assigned to the v1perc percent lowest flux 
    v2perc: the other end of the colormap assigned to the v2perc percent highest flux    
    '''
    #this makes two horizontal figures 
    fig,axs = plt.subplots(1,2,figsize=(12,6))
    # make sure image is an np array
    nimage = np.array(image)
    if (logscale):
        # determine the pixel values at the 10th and 95th percentile
        #v1 = scoreatpercentile(nimage,v1perc)
        #v2 = scoreatpercentile(nimage,v2perc)
        # display using imshow
        #
        # you can play with alternate cmaps in the function below, such as "viridis" or "gray"
        # The 'gray_r' color map reverses the color-scale so that dark display pixels are the brightest in the image
        #
        # vmin and vmax set the min and max pixel values that
        # will be mapped to the extremes of the colormap
        #print(f"v1 = {v1:5.2f}, v2 = {v2:5.2f}")
        #plot the two figures side by side
        v1 = 0.0001 # debug custom values
        v2 = 0.3 # debug custom values
        im0 = axs[0].imshow(nimage,cmap='gray_r',aspect='equal',norm=colors.LogNorm(vmin=v1,vmax=v2),origin='lower')
        im1 = axs[1].imshow(nimage,cmap='gray_r',aspect='equal',norm=colors.LogNorm(vmin=v1,vmax=v2),origin='lower')
    else:
        v1 = v1perc
        v2 = v2perc
        print(v1,v2)
        im0 = axs[0].imshow(nimage,cmap='gray_r',aspect='equal',vmin=v1,vmax=v2,origin='lower')
        im1 = axs[1].imshow(nimage,cmap='gray_r',aspect='equal',vmin=v1,vmax=v2,origin='lower')

    #plot colorbars for each subplot
    cbar0 = fig.colorbar(im0, ax=axs[0], fraction=0.04)
    cbar0.ax.set_ylabel('counts')
    cbar1 = fig.colorbar(im1, ax=axs[1], fraction=0.04)
    cbar1.ax.set_ylabel('counts')
    return fig,axs

In [114]:
## FROM Surface Photometry Lab
# Plots outputs of ellipse fitting
def plotellipse(isolist):
    plt.figure(figsize=(8, 8))
    plt.subplots_adjust(hspace=0.35, wspace=0.35)

    #plot the semi-major axis vs. the ellipticity
    plt.subplot(2, 2, 1)
    plt.errorbar(isolist.sma, isolist.eps, yerr=isolist.ellip_err,
                 fmt='o', markersize=4)
    plt.xlabel('Semimajor Axis Length (pix)')
    plt.ylabel('Ellipticity')

    #plot the semi-major axis vs. the PA
    plt.subplot(2, 2, 2)
    plt.errorbar(isolist.sma, isolist.pa / np.pi * 180.0,
                 yerr=isolist.pa_err / np.pi * 80.0, fmt='o', markersize=4)
    plt.xlabel('Semimajor Axis Length (pix)')
    plt.ylabel('PA (deg)')

    #plot the semi-major axis vs. the x center of that ellipse
    plt.subplot(2, 2, 3)
    plt.errorbar(isolist.sma, isolist.x0, yerr=isolist.x0_err, fmt='o',
                 markersize=4)
    plt.xlabel('Semimajor Axis Length (pix)')
    plt.ylabel('x0')

    #plot the semi-major axis vs. the y center of that ellipse
    plt.subplot(2, 2, 4)
    plt.errorbar(isolist.sma, isolist.y0, yerr=isolist.y0_err, fmt='o',
                 markersize=4)
    plt.xlabel('Semimajor Axis Length (pix)')
    plt.ylabel('y0')

In [115]:
## FROM Surface Photometry Lab
#this subroutine will plot every other isophote on top of the galaxy image
def plotaper(galname, image, isophote_list,vminperc=50,vmaxperc=99.99):

    #plot image
    #plt.subplots(figsize=(6,6))
    fig,ax = imdisplay2(image,v1perc=vminperc,v2perc=vmaxperc,logscale=True)

    #Loop through all apertures starting at the first (not zero) and going to every other aperture
    for i in range(1, len(isophote_list.sma),2):
        #create an elliptical aperture with the the correct geometry.  
        #Task is described here https://photutils.readthedocs.io/en/stable/api/photutils.aperture.EllipticalAperture.html#photutils.aperture.EllipticalAperture
        aperiter = EllipticalAperture((isophote_list.x0[i], isophote_list.y0[i]), isophote_list.sma[i], 
                                      isophote_list.sma[i] * (1-isophote_list.eps[i]), isophote_list.pa[i])
    
        #overplot apertures
        aperiter.plot(ax=ax[0],color='red')

    #plot the isophotes that have parameters fixed to the median parameters of all but the "apmin" inner isphotes
    #compute median isophote parameters ignoring inner isophotes.  This is because the inner isophotes all tend to be rather circular
    apmin = 20   #the minimum inner isophote in terms of isophote number.
    xcenav = np.median(isophote_list.x0[apmin:])
    ycenav = np.median(isophote_list.y0[apmin:])
    epsav = np.median(isophote_list.eps[apmin:])
    paav = np.median(isophote_list.pa[apmin:])
    
    #fig,ax = imdisplay(image,v1perc=vminperc,v2perc=vmaxperc,logscale=True)
    for i in range(1, len(isophote_list.sma),2):
        #create an elliptical aperture with the the correct geometry.  
        #Task is described here https://photutils.readthedocs.io/en/stable/api/photutils.aperture.EllipticalAperture.html#photutils.aperture.EllipticalAperture
        aperiter = EllipticalAperture((xcenav, ycenav), isophote_list.sma[i], 
                                      isophote_list.sma[i] * (1-epsav), paav)
    
        #overplot apertures
        aperiter.plot(ax=ax[1],color='blue')

    ax[0].set_title(galname + ' free ellipse parameters')
    ax[1].set_title(galname + ' fixed ellipse parameters')

    return xcenav, ycenav, epsav, paav

### Full Ellipse Function

In [116]:
from photutils.isophote import Ellipse
from photutils.isophote import EllipseGeometry
from photutils.aperture import EllipticalAperture
def ellipsefitting(gal, plot_opt, image_hdu=1):
    # TAKE IN:
    # gal : Galaxy row in catalog, with all initial guesses n stuff, also filename for image
    # image_hdu : which hdu is the image in hehe
    # plot_opt : do you want to do plots? True / False
    image = []
    header = []

    print(f'Galaxy name: {gal['name']}\tGalaxy filepath:{gal['file']}')
    
    with fits.open(gal["file"]) as hdul:
        # initial file image readin and header readin
        image = hdul[image_hdu].data
        header = hdul[image_hdu].header
        if plot_opt: # If true plot stuff
            # display first image
            imdisplay(image)
            
        # init empty geometry object for ELlipseGeometry
        geometry = {}
        geometry = EllipseGeometry(x0 = gal['xcen'], 
                                      y0 = gal['ycen'],
                                      sma = gal['sma'],
                                      eps = gal['ellip'],
                                      pa = gal['PA'] * np.pi/180)   #convert into radians
        print(geometry.sma,geometry.eps )
        
        #make an aperture to overplot so that we can check if it makes sens
        aper = EllipticalAperture((geometry.x0, geometry.y0), geometry.sma, 
                          geometry.sma * (1.-geometry.eps), geometry.pa)
        
        if plot_opt: # If true plot stuff
            #plot our image
            fig,ax = imdisplay(image,v1perc=50,v2perc=99.99,logscale=True)
            aper.plot(color='red') # plot aperture on top
            plt.title(gal['name'])

        #initialize dictionary for ellipse fitting
        ellipse = {}
        isolist = {}
        xcenav = {}
        ycenav = {}
        epsav = {}
        paav = {}
    return


## Background Removal

This section should cover background removal and be optional to the main script. Once we remove background, the script to find isochromes and fit a radial light profile to it should run on the background removed fits' from this section. To reproduce work, do background removal before main work.

In [16]:
#initialize dictionaries that contains names of images and headers, etc. 
imname = {}
data = {}
header = {}
corners = {}
bkgdim = {}
bkgmode = {}
bkgmed = {}
data_bsub = {}
imname_busb = {} 


def imdisplay(image, v1perc=10, v2perc=95, logscale=True):
    '''
    display an image 
    OPTIONAL KEYWORD PARAMETERS
    v1perc: one end of the colormap assigned to the v1perc percent lowest flux 
    v2perc: the other end of the colormap assigned to the v2perc percent highest flux    
    '''
    fig,ax = plt.subplots(figsize=(6,6))
    # make sure image is an np array
    nimage = np.array(image)
    # determine the pixel values at the 10th and 95th percentile
    v1 = scoreatpercentile(nimage,v1perc)
    v2 = scoreatpercentile(nimage,v2perc)
    # display using imshow
    #
    # you can play with alternate cmaps in the function below, such as "viridis" or "gray"
    # The 'gray_r' color map reverses the color-scale so that dark display pixels are the brightest in the image
    #
    # vmin and vmax set the min and max pixel values that
    # will be mapped to the extremes of the colormap
    print(v1,v2)
    if (logscale):
        im = ax.imshow(nimage,cmap='gray_r',aspect='equal',norm=colors.LogNorm(vmin=v1,vmax=v2),origin='lower')
    else:
        im = ax.imshow(nimage,cmap='gray_r',aspect='equal',vmin=v1,vmax=v2,origin='lower')
        
    cbar = fig.colorbar(im, ax=ax, fraction=0.04)
    cbar.ax.set_ylabel('counts')
    return fig,ax

def background_measure(imname,data,header,fig,ax,corners,bkg_size=5):
    fig, ax = imdisplay(data,v1perc=1,v2perc=99,logscale=True)
    plt.title(imname)
    print('image xdim = ', header['NAXIS1'], '; image ydim = ',header['NAXIS2'])
    reg = np.array([])
    reg = np.append(reg, data[corners[0][0]:bkg_size-1, corners[0][1]:bkg_size-1])   
    reg = np.append(reg, data[corners[1][0]:bkg_size-1, corners[1][1]:])  
    reg = np.append(reg, data[corners[2][0]:, corners[2][1]:])    
    reg = np.append(reg, data[corners[3][0]:, corners[3][1]:bkg_size-1]) 
    bkgmode =stats.mode(reg,keepdims=False)[0]
    bkgmed = np.median(reg)
    print('mode = ',bkgmode)
    print('median = ',bkgmed)
    rectlist = []
    for corn in corners:
        rect = plt.Rectangle((corn),bkg_size, bkg_size,linewidth=2,angle=0.0)
        rectlist.append(rect)
        pc = PatchCollection(rectlist,linewidth=1,edgecolor='r',facecolor='none')
        ax.add_collection(pc)
        
    return bkgmode,bkgmed



#A routine to plot the images with the background measurment corners shown in red 
def bkgregplot(data,header,imname, bkgdim, fig, ax):
    corners = []
    corners.append([0,0])
    corners.append([0,header['NAXIS2']-bkgdim])
    corners.append([header['NAXIS1']-bkgdim, header['NAXIS2']-bkgdim])
    corners.append([header['NAXIS1']-bkgdim,0])
    bkgmode,bkgmed = background_meas(imname, data, header, fig, ax, 
                                                       corners,bkg_size=bkgdim)
    return bkgmode, bkgmed


def plotwithredsquare(data,imname,header,bkgdim,fig,ax):
    data[galname], header[galname] = fits.getdata(imname[galname], header=True)
    bkgdim[galname] = 200 
    bkgmode[galname],bkgmed[galname] = bkgregplot(
    data[galname], header[galname], imname[galname], bkgdim[galname], fig, ax)

def bkgsubplot(data,header,imname,fix,ax): 
    data_bsub[galname] = data[galname] - bkgmed[galname] 
    fig,ax = imdisplay(data_bsub[galname],v1perc=40,v2perc=99.9,logscale=True)
    plt.title(galname + ' background reduced')
    imname_busb[galname] = imname[galname].replace("g.fits.fz","g_bsub.fits.fz")
    fits.writeto(imname_busb[galname],data_bsub[galname],overwrite=True,header=header[galname])